In [55]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingRandomSearchCV
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline



In [34]:
df = pd.read_csv("dados_usados/dados_para_modelo.csv")

In [35]:
df.head(10)

,status_do_emprestimo,valor_do_emprestimo,prazo_do_emprestimo_meses,taxa_de_juros,valor_da_parcela,tempo_de_emprego_anos,renda_anual,relacao_divida_renda,contas_de_credito_abertas,registros_publicos_negativos,...,finalidade_do_emprestimo_major_purchase,finalidade_do_emprestimo_medical,finalidade_do_emprestimo_moving,finalidade_do_emprestimo_other,finalidade_do_emprestimo_renewable_energy,finalidade_do_emprestimo_small_business,finalidade_do_emprestimo_vacation,finalidade_do_emprestimo_wedding,tipo_de_aplicacao_INDIVIDUAL,tipo_de_aplicacao_JOINT
0,1,10000.0,36,11.44,329.48,10.0,117000.0,26.24,16.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
1,1,8000.0,36,11.99,265.68,4.0,65000.0,22.05,17.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,1,15600.0,36,10.49,506.97,0.0,43057.0,12.79,13.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,1,7200.0,36,6.49,220.65,6.0,54000.0,2.60,6.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,0,24375.0,60,17.27,609.33,9.0,55000.0,33.95,13.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
5,1,20000.0,36,13.33,677.07,10.0,86788.0,16.31,8.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
6,1,18000.0,36,5.32,542.07,2.0,125000.0,1.36,8.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
7,1,13000.0,36,11.14,426.47,10.0,46000.0,26.87,11.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
8,1,18900.0,60,10.99,410.84,10.0,103000.0,12.52,13.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
9,1,26300.0,36,16.29,928.40,3.0,115000.0,23.69,13.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [36]:
x = df.drop("status_do_emprestimo", axis=1)
y = df["status_do_emprestimo"]

In [37]:
x.info()

<class 'pandas.DataFrame'>
RangeIndex: 383421 entries, 0 to 383420
Data columns (total 34 columns):
 #   Column                                       Non-Null Count   Dtype  
---  ------                                       --------------   -----  
 0   valor_do_emprestimo                          383421 non-null  float64
 1   prazo_do_emprestimo_meses                    383421 non-null  int64  
 2   taxa_de_juros                                383421 non-null  float64
 3   valor_da_parcela                             383421 non-null  float64
 4   tempo_de_emprego_anos                        383421 non-null  float64
 5   renda_anual                                  383421 non-null  float64
 6   relacao_divida_renda                         383421 non-null  float64
 7   contas_de_credito_abertas                    383421 non-null  float64
 8   registros_publicos_negativos                 383421 non-null  float64
 9   saldo_de_credito_rotativo                    383421 non-null  float64


In [38]:
y.value_counts()

status_do_emprestimo
1    308372
0     75049
Name: count, dtype: int64

In [65]:
x_train, x_test, y_train, y_test= train_test_split(x, y, test_size=0.2, stratify= y, random_state=42)

In [40]:
len(x_train), len(y_train)

(306736, 306736)

In [41]:
len(x_test), len(y_test)

(76685, 76685)

In [42]:
pipeline_logistic = Pipeline(
   [ ("scaler", StandardScaler()),
    ("model", LogisticRegression())]
)

pipeline_knn = Pipeline(
    [("scaler", StandardScaler()),
    ("model", KNeighborsClassifier())]
)

In [43]:
modelos = {
    "KNN" : pipeline_knn,
    "Logistic Regressionr" : pipeline_logistic,
    "Ramdom Forest" : RandomForestClassifier(random_state=42),
    "XGBoosting" : XGBClassifier(random_state=42),
    "Lightgbm" : LGBMClassifier(random_state=42)
}

In [64]:
avaliacao = {}

for name, modelo in modelos.items():
    cross_score = cross_val_score(
        modelo, x_train, y_train,
        cv=5,
        scoring= "roc_auc"
    )
    avaliacao[name] = np.mean(cross_score)
    print(f"{name} : {avaliacao[name]}")


KNN : 0.6119300028983286
Logistic Regressionr : 0.7026818851274619
Ramdom Forest : 0.6965215206081946
XGBoosting : 0.7101020007130581
[LightGBM] [Info] Number of positive: 172688, number of negative: 42027
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.014882 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1943
[LightGBM] [Info] Number of data points in the train set: 214715, number of used features: 33
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.804266 -> initscore=1.413174
[LightGBM] [Info] Start training from score 1.413174
[LightGBM] [Info] Number of positive: 172688, number of negative: 42027
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005716 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1943
[LightGBM] [Info] Number of data points 

In [67]:
knn_params = {
  "model__n_neighbors" : [1, 3, 5, 7, 9, 20, 30, 50],
  "model__weights" : ["uniform", "distance"]
}

random_forest_params = {
    "n_estimators" : np.arange(300, 1500, 90),
    "max_depth" : [1, 5, 9, 20, 50],
    "min_samples_leaf" : np.arange(1, 10, 3),
    "min_samples_split" : np.arange(1, 15, 3)
}

logistic_params = {
    "model__C" : np.logspace(-3, 3, 10),
    "model__solver" : ["lbfgs", "liblinear"]
}

gradient_boosting_params = {
    "n_estimators" : np.arange(100, 1080, 80),
    "max_depth" : [1, 3, 9, 30, 50],
    "subsample" : [0.2, 0.5, 0.7, 1],
    "min_child_weight" : np.arange(1, 10, 3),
}

light_params = {
    "num_leaves" : [31, 63, 127],
    "n_estimators" : [100, 300, 500, 1000],
    "max_depth" : [1, 5, 9, 20],
    "subsample" : [0.2, 0.7, 1],
    "colsample_bytree" : [0.2, 0.5, 1],
    "learning_rate" : [0.2, 0.5, 1]
}



In [46]:
x_sample,_, y_sample,_ = train_test_split(x_train, y_train, train_size=100000, stratify=y_train, random_state=42)

In [ ]:
def test_randomized_search(model, params, x_sample, y_sample):
    randomized = HalvingRandomSearchCV(
    model,
    params,
    cv= 5,
    n_candidates = 50,
    factor=3,
    scoring="roc_auc",
    n_jobs = -1,
    min_resources= 3000,
    random_state=42,
    verbose=2
)

    randomized.fit(x_sample, y_sample)
    randomized
    print(f"O melhor score foi: {randomized.best_score_}")
    print(f"Os melhores parametros foram: {randomized.best_params_}")


In [48]:
knn_modelo = test_randomized_search(pipeline_knn, knn_params, x_train, y_train)

n_iterations: 3
n_required_iterations: 3
n_possible_iterations: 5
min_resources_: 3000
max_resources_: 306736
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 16
n_resources: 3000
Fitting 5 folds for each of 16 candidates, totalling 80 fits


/home/Erick/miniconda3/envs/data/lib/python3.11/site-packages/sklearn/model_selection/_search.py:324: UserWarning: The total space of parameters 16 is smaller than n_iter=50. Running 16 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


[CV] END .......model__n_neighbors=1, model__weights=uniform; total time=   0.1s
[CV] END ......model__n_neighbors=1, model__weights=distance; total time=   0.1s[CV] END .......model__n_neighbors=1, model__weights=uniform; total time=   0.1s

[CV] END ......model__n_neighbors=1, model__weights=distance; total time=   0.1s
[CV] END .......model__n_neighbors=1, model__weights=uniform; total time=   0.1s
[CV] END ......model__n_neighbors=1, model__weights=distance; total time=   0.1s
[CV] END .......model__n_neighbors=1, model__weights=uniform; total time=   0.1s
[CV] END .......model__n_neighbors=1, model__weights=uniform; total time=   0.1s
[CV] END ......model__n_neighbors=1, model__weights=distance; total time=   0.0s
[CV] END ......model__n_neighbors=1, model__weights=distance; total time=   0.0s
[CV] END .......model__n_neighbors=3, model__weights=uniform; total time=   0.0s
[CV] END .......model__n_neighbors=3, model__weights=uniform; total time=   0.0s
[CV] END .......model__n_nei

In [49]:
logistic_modelo = test_randomized_search(pipeline_logistic, logistic_params, x_train, y_train)

n_iterations: 3
n_required_iterations: 3
n_possible_iterations: 5
min_resources_: 3000
max_resources_: 306736
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 20
n_resources: 3000
Fitting 5 folds for each of 20 candidates, totalling 100 fits


/home/Erick/miniconda3/envs/data/lib/python3.11/site-packages/sklearn/model_selection/_search.py:324: UserWarning: The total space of parameters 20 is smaller than n_iter=50. Running 20 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


[CV] END ................model__C=0.001, model__solver=lbfgs; total time=   0.0s
[CV] END ................model__C=0.001, model__solver=lbfgs; total time=   0.0s
[CV] END ................model__C=0.001, model__solver=lbfgs; total time=   0.0s
[CV] END ................model__C=0.001, model__solver=lbfgs; total time=   0.0s
[CV] END ................model__C=0.001, model__solver=lbfgs; total time=   0.0s
[CV] END ............model__C=0.001, model__solver=liblinear; total time=   0.0s
[CV] END ............model__C=0.001, model__solver=liblinear; total time=   0.0s
[CV] END ............model__C=0.001, model__solver=liblinear; total time=   0.0s
[CV] END ............model__C=0.001, model__solver=liblinear; total time=   0.0s
[CV] END ............model__C=0.001, model__solver=liblinear; total time=   0.0s
[CV] END .model__C=0.004641588833612777, model__solver=lbfgs; total time=   0.0s
[CV] END .model__C=0.004641588833612777, model__solver=lbfgs; total time=   0.0s
[CV] END .model__C=0.0046415

In [50]:
gradient_modelo = test_randomized_search(XGBClassifier(), gradient_boosting_params, x_train, y_train)

n_iterations: 4
n_required_iterations: 4
n_possible_iterations: 5
min_resources_: 3000
max_resources_: 306736
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 50
n_resources: 3000
Fitting 5 folds for each of 50 candidates, totalling 250 fits
[CV] END max_depth=30, min_child_weight=4, n_estimators=900, subsample=1; total time=   3.1s
[CV] END max_depth=30, min_child_weight=4, n_estimators=900, subsample=1; total time=   3.1s[CV] END max_depth=30, min_child_weight=4, n_estimators=900, subsample=1; total time=   3.1s

[CV] END max_depth=1, min_child_weight=7, n_estimators=420, subsample=0.2; total time=   0.2s
[CV] END max_depth=30, min_child_weight=1, n_estimators=900, subsample=1; total time=   3.4s
[CV] END max_depth=30, min_child_weight=1, n_estimators=900, subsample=1; total time=   3.4s
[CV] END max_depth=30, min_child_weight=1, n_estimators=900, subsample=1; total time=   3.4s
[CV] END max_depth=30, min_child_weight=1, n_estimators=900, subsample=1; total ti

In [ ]:
light_modelo = test_randomized_search(LGBMClassifier(), light_params, x_train, y_train)

n_iterations: 3
n_required_iterations: 3
n_possible_iterations: 5
min_resources_: 3000
max_resources_: 306736
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 20
n_resources: 3000
Fitting 5 folds for each of 20 candidates, totalling 100 fits
[LightGBM] [Info] Number of positive: 1898, number of negative: 502
[LightGBM] [Info] Number of positive: 1902, number of negative: 498
[LightGBM] [Info] Number of positive: 1937, number of negative: 462
[LightGBM] [Info] Number of positive: 1937, number of negative: 462
[LightGBM] [Info] Number of positive: 1893, number of negative: 507
[LightGBM] [Info] Number of positive: 1918, number of negative: 482
[LightGBM] [Info] Number of positive: 1893, number of negative: 507
[LightGBM] [Info] Number of positive: 1902, number of negative: 498
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.164151 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins

In [ ]:
random_forest_modelo = test_randomized_search(RandomForestClassifier(), random_forest_params, x_train, y_train)